<a href="https://colab.research.google.com/github/mayway500/FPE/blob/main/US%26Malayisa_Env_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install furl
!pip install html5lib==1.0b8
!pip install html-table-parser-python3
!pip install sdmx1
!pip install sdmx1[cache]
!pip install sdmx1[cache,docs,tests]


In [12]:
import pandas as pd

def OPEC():
    sheet_id = '1GIww6W3wwn7iQJ7wt9ugwwYRJ7szjQRN'
    gid = '1605979640'

    # Corrected Google Sheets export URL format
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&gid={gid}"
    try:
        # Read the CSV without a header, assuming the first row is data and columns 0 and 1 are Date and OPEC
        raw_opec_data = pd.read_csv(url, header=None)
        print("--- Debug: After initial CSV read ---")
        print(f"Raw data shape: {raw_opec_data.shape}")
        print("Raw data head:")
        print(raw_opec_data.head())

        # Check if data has enough columns and manually assign column names
        if raw_opec_data.shape[1] >= 2:
            all_opec_data = raw_opec_data[[0, 1]].copy() # Select only the first two columns
            all_opec_data.columns = ['Date', 'OPEC'] # Assign new column names
        else:
            print("Warning: Not enough columns found in OPEC data. Expected at least 2.")
            return pd.DataFrame(columns=['Date', 'OPEC']).set_index('Date')

        print("--- Debug: After column selection and naming ---")
        print(f"All OPEC data shape: {all_opec_data.shape}")
        print("All OPEC data head:")
        print(all_opec_data.head())

        # Remove the first row which contains metadata ('Attribute:data', 'Attribute:val')
        all_opec_data = all_opec_data.iloc[1:].copy()
        print("--- Debug: After removing metadata row ---")
        print(f"All OPEC data shape: {all_opec_data.shape}")
        print("All OPEC data head:")
        print(all_opec_data.head())

        # Convert 'Date' column to datetime objects with explicit format to avoid UserWarning
        # Corrected date format to '%m/%d/%Y'
        all_opec_data['Date'] = pd.to_datetime(all_opec_data['Date'], format='%m/%d/%Y', errors='coerce')
        print("--- Debug: After Date conversion ---")
        print(f"All OPEC data shape: {all_opec_data.shape}")
        print("All OPEC data head:")
        print(all_opec_data.head())

        # Drop rows where Date is NaT or OPEC price is NaN (after converting to numeric)
        all_opec_data['OPEC'] = pd.to_numeric(all_opec_data['OPEC'], errors='coerce')
        all_opec_data.dropna(subset=['Date', 'OPEC'], inplace=True)
        print("--- Debug: After dropping NaNs ---")
        print(f"All OPEC data shape: {all_opec_data.shape}")
        print("All OPEC data head:")
        print(all_opec_data.head())

        # Handle duplicate dates by taking the mean before setting the index
        if not all_opec_data['Date'].is_unique:
            all_opec_data = all_opec_data.groupby('Date')['OPEC'].mean().reset_index()
            print("Resolved duplicate dates in OPEC data by taking the mean.")

        all_opec_data = all_opec_data.set_index('Date')
        all_opec_data.sort_index(inplace=True)

        # --- START Modification to inspect raw data date range ---
        print(f"Raw data date range before 2002-01-01 filter: {all_opec_data.index.min()} to {all_opec_data.index.max()}")
        print("Head of data before 2002-01-01 filter:")
        print(all_opec_data.head())
        # --- END Modification ---

        # Explicitly filter data to be from 2002 onwards
        all_opec_data = all_opec_data[all_opec_data.index >= '2002-01-01']

        # Add check for empty DataFrame after filtering
        if all_opec_data.empty:
            print("No OPEC data available from 2002-01-01 onwards after filtering. Returning empty DataFrame.")
            return pd.DataFrame(columns=['OPEC'], index=pd.Index([], name='Date'))

        min_date = all_opec_data.index.min()
        max_date = all_opec_data.index.max()
        if pd.isna(min_date) or pd.isna(max_date):
            print("Warning: Could not determine date range for OPEC reindexing.")
        else:
            full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')
            all_opec_data = all_opec_data.reindex(full_date_range)


        print(all_opec_data)
        return all_opec_data

    except Exception as e:
        print(f"Error loading OPEC data: {e}")
        return pd.DataFrame(columns=['Date', 'OPEC']).set_index('Date')

In [14]:
Opecdata=OPEC()

--- Debug: After initial CSV read ---
Raw data shape: (5983, 3)
Raw data head:
                0              1   2
0  Attribute:data  Attribute:val NaN
1        1/2/2003          30.05 NaN
2        1/3/2003          30.83 NaN
3        1/6/2003          30.71 NaN
4        1/7/2003          29.72 NaN
--- Debug: After column selection and naming ---
All OPEC data shape: (5983, 2)
All OPEC data head:
             Date           OPEC
0  Attribute:data  Attribute:val
1        1/2/2003          30.05
2        1/3/2003          30.83
3        1/6/2003          30.71
4        1/7/2003          29.72
--- Debug: After removing metadata row ---
All OPEC data shape: (5982, 2)
All OPEC data head:
       Date   OPEC
1  1/2/2003  30.05
2  1/3/2003  30.83
3  1/6/2003  30.71
4  1/7/2003  29.72
5  1/8/2003  28.86
--- Debug: After Date conversion ---
All OPEC data shape: (5982, 2)
All OPEC data head:
        Date   OPEC
1 2003-01-02  30.05
2 2003-01-03  30.83
3 2003-01-06  30.71
4 2003-01-07  29.72
5 200

In [15]:
import pandas as pd
import json
import requests
import os
import numpy as np
from google.colab import drive
from datetime import datetime
from urllib.parse import quote
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
from google.colab import data_table
from datetime import timedelta, date
from sklearn.linear_model import LinearRegression
import urllib.request
# Removed: from html_table_parser.parser import HTMLTableParser
# Removed: from furl import furl

import torch # Import torch for tensor conversion


# IMPORTANT: To resolve 'ModuleNotFoundError: No module named 'sdmx'', please ensure
# the initial dependency installation cell (Mw_z8_-gap2i) has been successfully executed.
# If you are still facing the error, run '!pip install sdmx1' in a new code cell before executing this block.


class DataFetcher:
    def __init__(self):
        self.todays_date = datetime.now().date()
        self.base_save_path = '/content/drive/MyDrive/deep learning codes/EIAAPI_DOWNLOAD'
        self.merged_data_path = os.path.join(self.base_save_path, 'solutions', 'mergedata')
        os.makedirs(self.base_save_path, exist_ok=True)
        os.makedirs(self.merged_data_path, exist_ok=True)
        # try:
        #     drive.mount('/content/drive')
        # except:
        #     print("Google Drive already mounted or failed to mount.")

    def WTI(self):
        Gh1files = []
        # Corrected the start date in the base URL to 2002-01-01
        url = "https://api.eia.gov/v2/petroleum/pri/spt/data/?frequency=daily&data[0]=value&facets[series][]=RWTC&start=2002-01-01&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000&api_key=7e361b53e231aaf90ac6dcd91c0dc07d"
        headers = {"api_key": "7e361b53e231aaf90ac6dcd91c0dc07d", "host": "api.eia.gov"}

        year = 2002
        while year < self.todays_date.year:
            year += 1

            querystring = {'frequency': 'daily', 'data[0]': 'value', 'start': f'{year}-01-01', 'end': f'{year}-12-31', 'offset': '0', 'length': '5000'}

            response = requests.request("GET", url, headers=headers, params=querystring)
            if response.status_code != 200:
                print(f"Error: API request failed for WTI year {year} with status code {response.status_code}")
                print(f"Response text: {response.text}")
                continue

            output_path = os.path.join(self.base_save_path, f"{year}pricesWTI_year.json")
            with open(output_path, "w") as outfile:
                json.dump(response.json(), outfile)

            Gh1files.append(response.json())

        # Reorganizing data from saved JSON files
        serK = []
        serH = []
        year = 2002
        while year < self.todays_date.year:
            year += 1
            output_path = os.path.join(self.base_save_path, f"{year}pricesWTI_year.json")
            try:
                with open(output_path, "r") as file:
                    DData = json.load(file)
                    if "response" in DData and "data" in DData["response"]:
                        for data_item in DData["response"]["data"]:
                            serK.append(data_item['period'])
                            serH.append(data_item['value'])
            except FileNotFoundError:
                print(f"Warning: WTI Data file for year {year} not found. Skipping.")
                continue
            except json.JSONDecodeError:
                print(f"Warning: Error decoding JSON for WTI year {year}. Skipping.")
                continue

        if serK and serH:
            WTIPrice = pd.DataFrame({'Date': serK, 'WTI': serH})
            WTIPrice['Date'] = pd.to_datetime(WTIPrice['Date'])
            WTIPrice = WTIPrice.set_index('Date').sort_index()

            # Explicitly filter data to be from 2002 onwards
            WTIPrice = WTIPrice[WTIPrice.index >= '2002-01-01']


            full_date_range = pd.date_range(start=WTIPrice.index.min(), end=WTIPrice.index.max(), freq="D")
            WWWTIPrice = WTIPrice.reindex(full_date_range)
            WWWTIPrice.index.name = 'Date' # Ensure index name is 'Date'

            print(WWWTIPrice)
            return WWWTIPrice
        else:
            print("No WTI data was loaded.")
            return pd.DataFrame()




    def OPEC(self):
        sheet_id = '1GIww6W3wwn7iQJ7wt9ugwwYRJ7szjQRN'
        gid = '1605979640'

        # Corrected Google Sheets export URL format
        url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&gid={gid}"
        try:
            # Read the CSV without a header, assuming the first row is data and columns 0 and 1 are Date and OPEC
            raw_opec_data = pd.read_csv(url, header=None)
            print("--- Debug: After initial CSV read ---")
            print(f"Raw data shape: {raw_opec_data.shape}")
            print("Raw data head:")
            print(raw_opec_data.head())

            # Check if data has enough columns and manually assign column names
            if raw_opec_data.shape[1] >= 2:
                all_opec_data = raw_opec_data[[0, 1]].copy() # Select only the first two columns
                all_opec_data.columns = ['Date', 'OPEC'] # Assign new column names
            else:
                print("Warning: Not enough columns found in OPEC data. Expected at least 2.")
                return pd.DataFrame(columns=['Date', 'OPEC']).set_index('Date')

            print("--- Debug: After column selection and naming ---")
            print(f"All OPEC data shape: {all_opec_data.shape}")
            print("All OPEC data head:")
            print(all_opec_data.head())

            # Remove the first row which contains metadata ('Attribute:data', 'Attribute:val')
            all_opec_data = all_opec_data.iloc[1:].copy()
            print("--- Debug: After removing metadata row ---")
            print(f"All OPEC data shape: {all_opec_data.shape}")
            print("All OPEC data head:")
            print(all_opec_data.head())

            # Convert 'Date' column to datetime objects with explicit format to avoid UserWarning
            # Corrected date format to '%m/%d/%Y'
            all_opec_data['Date'] = pd.to_datetime(all_opec_data['Date'], format='%m/%d/%Y', errors='coerce')
            print("--- Debug: After Date conversion ---")
            print(f"All OPEC data shape: {all_opec_data.shape}")
            print("All OPEC data head:")
            print(all_opec_data.head())

            # Drop rows where Date is NaT or OPEC price is NaN (after converting to numeric)
            all_opec_data['OPEC'] = pd.to_numeric(all_opec_data['OPEC'], errors='coerce')
            all_opec_data.dropna(subset=['Date', 'OPEC'], inplace=True)
            print("--- Debug: After dropping NaNs ---")
            print(f"All OPEC data shape: {all_opec_data.shape}")
            print("All OPEC data head:")
            print(all_opec_data.head())

            # Handle duplicate dates by taking the mean before setting the index
            if not all_opec_data['Date'].is_unique:
                all_opec_data = all_opec_data.groupby('Date')['OPEC'].mean().reset_index()
                print("Resolved duplicate dates in OPEC data by taking the mean.")

            all_opec_data = all_opec_data.set_index('Date')
            all_opec_data.sort_index(inplace=True)

            # --- START Modification to inspect raw data date range ---
            print(f"Raw data date range before 2002-01-01 filter: {all_opec_data.index.min()} to {all_opec_data.index.max()}")
            print("Head of data before 2002-01-01 filter:")
            print(all_opec_data.head())
            # --- END Modification ---

            # Explicitly filter data to be from 2002 onwards
            all_opec_data = all_opec_data[all_opec_data.index >= '2002-01-01']

            # Add check for empty DataFrame after filtering
            if all_opec_data.empty:
                print("No OPEC data available from 2002-01-01 onwards after filtering. Returning empty DataFrame.")
                return pd.DataFrame(columns=['OPEC'], index=pd.Index([], name='Date'))

            min_date = all_opec_data.index.min()
            max_date = all_opec_data.index.max()
            if pd.isna(min_date) or pd.isna(max_date):
                print("Warning: Could not determine date range for OPEC reindexing.")
            else:
                full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')
                all_opec_data = all_opec_data.reindex(full_date_range)


            print(all_opec_data)
            return all_opec_data

        except Exception as e:
            print(f"Error loading OPEC data: {e}")
            return pd.DataFrame(columns=['Date', 'OPEC']).set_index('Date')


    def USAFuel_Daily(self):
        # Corrected the start date in the base URL to 2002-01-01
        url = "https://api.eia.gov/v2/petroleum/pri/spt/data/?frequency=daily&data[0]=value&facets[series][]=EER_EPD2DC_PF4_Y05LA_DPG&facets[series][]=EER_EPD2DXL0_PF4_RGC_DPG&facets[series][]=EER_EPD2DXL0_PF4_Y35NY_DPG&facets[series][]=EER_EPD2F_PF4_Y35NY_DPG&facets[series][]=EER_EPJK_PF4_RGC_DPG&facets[series][]=EER_EPLLPA_PF4_Y44MB_DPG&facets[series][]=EER_EPMRR_PF4_Y05LA_DPG&facets[series][]=EER_EPMRU_PF4_RGC_DPG&facets[series][]=EER_EPMRU_PF4_Y35NY_DPG&start=2002-01-01&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000&api_key=7e361b53e231aaf90ac6dcd91c0dc07d"
        headers = {"api_key": "7e361b53e231aaf90ac6dcd91c0dc07d", "host": "api.eia.gov"}
        Gh1files = []
        year = 2002
        while year <= self.todays_date.year:
            querystring = {'frequency': 'daily', 'data[0]': 'value', 'start': f'{year}-01-01', 'end': f'{year}-12-31', 'offset': '0', 'length': '5000'}
            response = requests.request("GET", url, headers=headers, params=querystring)
            if response.status_code != 200:
                print(f"Error: API request failed for Fuel Daily year {year} with status code {response.status_code}")
                print(f"Response text: {response.text}")
                year += 1
                continue

            output_path = os.path.join(self.base_save_path, f"{year}pricesFuel_year.json")
            with open(output_path, "w") as outfile:
                json.dump(response.json(), outfile)
            print(f"Successfully wrote data for year {year} to {output_path}")

            Gh1files.append(response.json())
            year += 1

        if Gh1files:
            all_fuel_records = []
            for year_data in Gh1files:
                if "response" in year_data and "data" in year_data["response"]:
                    for record in year_data["response"]["data"]:
                        all_fuel_records.append(record)

            if all_fuel_records:
                FuelPrice = pd.DataFrame(all_fuel_records)
                FuelPrice['period'] = pd.to_datetime(FuelPrice['period'])
                # Ensure the column for series ID exists and rename it
                if 'series' in FuelPrice.columns: # Assuming 'series' is the correct column name
                    FuelPrice.rename(columns={'period': 'Date', 'value': 'Price', 'series': 'Series'}, inplace=True)
                    # Drop rows where Date is NaT or Price is NaN
                    FuelPrice.dropna(subset=['Date', 'Price', 'Series'], inplace=True)

                    # Explicitly filter data to be from 2002 onwards
                    FuelPrice = FuelPrice[FuelPrice['Date'] >= '2002-01-01']


                    # Pivot the data to have dates as index and series as columns
                    FuelPrice_pivot = FuelPrice.pivot_table(index='Date', columns='Series', values='Price', aggfunc='first')

                    # Initialize FFFuelprice immediately after FuelPrice_pivot is created
                    FFFuelprice = FuelPrice_pivot

                    # Reindex to a full date range and forward fill missing values
                    min_date = FFFuelprice.index.min()
                    max_date = FFFuelprice.index.max()
                    if pd.isna(min_date) or pd.isna(max_date):
                        print("Warning: Could not determine date range for Fuel Daily reindexing.")
                        # FFFuelprice is already initialized, no need to reassign
                    else:
                        full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')
                        FFFuelprice = FFFuelprice.reindex(full_date_range)

                    FFFuelprice.ffill(inplace=True) # Updated fillna to ffill()
                    FFFuelprice.index.name = 'Date' # Ensure index name is 'Date'


                    merged_output_path = os.path.join(self.merged_data_path, "FuelMergeddata1.json")
                    try:
                        FFFuelprice.to_json(merged_output_path, orient='index', indent=4) # Save as index-oriented JSON
                        print(f"Successfully saved merged Fuel data to {merged_output_path}")
                    except Exception as e:
                        print(f"Error saving merged Fuel data: {e}")

                    print(FFFuelprice)
                    return FFFuelprice
                else:
                    print("Error: 'series' column not found in Fuel Daily data after fetching. Please check the API response structure.")
                    return pd.DataFrame() # Return empty DataFrame if 'series' is missing

            else:
                print("No data found in the Fuel Daily API responses.")
                return pd.DataFrame() # Return an empty DataFrame if no data is found
        else:
            print("No Fuel Daily data was loaded from API.")
            return pd.DataFrame() # Return an empty DataFrame if no files were processed

    def Brent(self):
        #Fetching the bulk data from website  and reorganizing on to google drive
        Gh1files=[]
        # Corrected the start date in the base URL to 2002-01-01
        url = "https://api.eia.gov/v2/petroleum/pri/spt/data/?frequency=daily&data[0]=value&facets[series][]=RBRTE&start=2002-01-01&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000&api_key=7e361b53e231aaf90ac6dcd91c0dc07d"
        headers = {"api_key":"7e361b53e231aaf90ac6dcd91c0dc07d","host":"api.eia.gov"}

        year = 2002
        while year < self.todays_date.year:
            year += 1

            querystring={'frequency':'daily','data[0]':'value','start':f'{year}-01-01','end':f'{year}-12-31','offset':'0','length':'5000'}

            response = requests.request("GET", url, headers=headers, params=querystring)
            if response.status_code != 200:
                print(f"Error: API request failed for Brent year {year} with status code {response.status_code}")
                print(f"Response text: {response.text}")
                continue

            output_path = os.path.join(self.base_save_path, f"{year}pricesBrent_year.json")
            with open(output_path, "w") as outfile:
                json.dump(response.json(), outfile)

            Gh1files.append(response.json())

        # Reorganizing data from saved JSON files
        serK = []
        serH = []
        year = 2002
        while year < self.todays_date.year:
            year += 1
            output_path = os.path.join(self.base_save_path, f"{year}pricesBrent_year.json")
            try:
                with open(output_path, "r") as file:
                    DData = json.load(file)
                    if "response" in DData and "data" in DData["response"]:
                        for data_item in DData["response"]["data"]:
                            serK.append(data_item['period'])
                            serH.append(data_item['value'])
            except FileNotFoundError:
                print(f"Warning: Brent Data file for year {year} not found. Skipping.")
                continue
            except json.JSONDecodeError:
                print(f"Warning: Error decoding JSON for Brent year {year}. Skipping.")
                continue

        if serK and serH:
            BrentPrice = pd.DataFrame({'Date': serK, 'Brent': serH})
            BrentPrice['Date'] = pd.to_datetime(BrentPrice['Date'])
            BrentPrice = BrentPrice.set_index('Date').sort_index()

            # Explicitly filter data to be from 2002 onwards
            BrentPrice = BrentPrice[BrentPrice.index >= '2002-01-01']


            full_date_range = pd.date_range(start=BrentPrice.index.min(), end=BrentPrice.index.max(), freq="D")
            BBBBrentPrice = BrentPrice.reindex(full_date_range)
            BBBBrentPrice.index.name = 'Date' # Ensure index name is 'Date'

            print(BBBBrentPrice)
            return BBBBrentPrice
        else:
            print("No Brent data was loaded.")
            return pd.DataFrame(columns=['Brent'], index=pd.Index([], name='Date'))

    def SDR_XDC(self):
        import sdmx
        import pandas as pd
        from datetime import date


        IMF_DATA = sdmx.Client('IMF_DATA')
        print("Attempting to query data for XDR_XDC (Period Average) from the 'ER' dataflow for multiple countries and monthly frequency...")


        dataflow_id_to_query = 'ER'
        key_to_query_dict = {
            'FREQUENCY': 'M',                  # Monthly frequency
            'COUNTRY': ['USA',  'MYS', ], # United States, Ghana, Malaysia, Eurozone
            'INDICATOR': 'XDR_XDC',            # Special Drawing Rights to XDC
            'TYPE_OF_TRANSFORMATION': 'PA_RT'  # Period Average Rate
        }

        params_to_query = {'startPeriod': 2010, 'endPeriod':  date.today()}

        print(f"Querying dataflow: {dataflow_id_to_query} with key dictionary: {key_to_query_dict} and params: {params_to_query}")
        data_msg = IMF_DATA.data(dataflow_id_to_query, key=key_to_query_dict, params=params_to_query)

            # Convert to pandas DataFrame
        er_data_df_multi = sdmx.to_pandas(data_msg)

        # Convert Series to DataFrame if it's a Series and ensure 'value' column
        if isinstance(er_data_df_multi, pd.Series):
            er_data_df_multi = er_data_df_multi.to_frame(name='value')
        elif er_data_df_multi.empty:
            # Ensure an empty DataFrame also has a 'value' column if it's expected later
            er_data_df_multi = pd.DataFrame(columns=['value'])

        # Ensure the TIME_PERIOD index is unique in the raw data before further processing
        if not er_data_df_multi.empty and 'TIME_PERIOD' in er_data_df_multi.index.names:
            # Remove duplicates based on the entire MultiIndex, keeping the first occurrence
            er_data_df_multi = er_data_df_multi.loc[~er_data_df_multi.index.duplicated(keep='first')]


        if not er_data_df_multi.empty:
            print(f"\nSuccessfully fetched data for {dataflow_id_to_query} with the specified key.\nDisplaying head of the DataFrame:")
            display(er_data_df_multi.head())

        # Optionally, check specific unique values for confirmation
        print("\nUnique countries in fetched data:", er_data_df_multi.index.get_level_values('COUNTRY').unique().tolist())
        print("Unique indicators in fetched data:", er_data_df_multi.index.get_level_values('INDICATOR').unique().tolist())
        print("Unique transformation types in fetched data:", er_data_df_multi.index.get_level_values('TYPE_OF_TRANSFORMATION').unique().tolist())
        print("Unique frequencies in fetched data:", er_data_df_multi.index.get_level_values('FREQUENCY').unique().tolist())

        return er_data_df_multi # Corrected return value










    def Malaysiafuelprice(self):
        print("Fetching Malaysia fuel price data...")
        # Corrected URL to a direct data endpoint for fuel prices in Malaysia
        url = "https://api.data.gov.my/data-catalogue?id=fuelprice" # Updated URL

        try:
            response = requests.get(url=url)
            response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
            response_json = response.json()

            extracted_data = []
            # Based on inspection, the API returns a direct list of dictionaries.
            if isinstance(response_json, list):
                for record in response_json:
                    # Check for required fields and add to extracted_data
                    if isinstance(record, dict) and 'date' in record and \
                       any(fuel_type in record for fuel_type in ['ron95', 'ron97', 'diesel', 'diesel_eastmsia']):
                        # Extract relevant fuel types and their prices
                        fuel_types_present = {k: v for k, v in record.items() if k in ['ron95', 'ron97', 'diesel', 'diesel_eastmsia']}

                        # Create a separate record for each fuel type to then pivot
                        for fuel_type, price in fuel_types_present.items():
                            extracted_data.append({
                                'Date': record['date'],
                                'Fuel_Type': fuel_type,
                                'Price': price
                            })
                    else:
                        print(f"Warning: Skipping malformed record: {record}")
            else:
                print("Error: Unexpected API response format. Expected a direct list of records.")
                return pd.DataFrame()

            if not extracted_data:
                print("No valid fuel price records found in the API response.")
                return pd.DataFrame()

            # Convert to Pandas DataFrame
            df = pd.DataFrame(extracted_data)

            # Convert 'Date' column to datetime objects
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
            df['Price'] = pd.to_numeric(df['Price'], errors='coerce') # Ensure price is numeric
            df.dropna(subset=['Date', 'Price'], inplace=True) # Drop rows with invalid dates or prices

            # Filter data from '2002-01-01' onwards
            df = df[df['Date'] >= '2002-01-01'].copy() # Use .copy() to avoid SettingWithCopyWarning

            if df.empty:
                print("No Malaysia fuel price data available from 2002-01-01 onwards.")
                return pd.DataFrame()

            # Pivot the DataFrame
            # Handle potential duplicates by taking the first price for a given date and fuel type
            malaysia_fuel_price_pivot = df.pivot_table(
                index='Date',
                columns='Fuel_Type',
                values='Price',
                aggfunc='first'
            )

            # Create a full date range and reindex
            min_date = malaysia_fuel_price_pivot.index.min()
            max_date = malaysia_fuel_price_pivot.index.max()
            full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')
            malaysia_fuel_price_reindexed = malaysia_fuel_price_pivot.reindex(full_date_range)

            # Forward-fill missing values
            malaysia_fuel_price_reindexed.ffill(inplace=True) # Updated fillna to ffill()

            # Ensure index name is 'Date'
            malaysia_fuel_price_reindexed.index.name = 'Date'

            print("Successfully processed Malaysia fuel price data.")
            print(malaysia_fuel_price_reindexed.head())
            return malaysia_fuel_price_reindexed

        except requests.exceptions.RequestException as e:
            print(f"Error fetching Malaysia fuel price data: {e}")
            return pd.DataFrame()
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON from Malaysia fuel price API: {e}")
            return pd.DataFrame()
        except Exception as e:
            print(f"An unexpected error occurred during Malaysia fuel price processing: {e}")
            return pd.DataFrame()

    def prepare_country_features(self, merged_df):
        print("Preparing country-specific features...")

        # 2. Dynamically identify all USA fuel price series (columns starting with 'EER_EP')
        usa_fuel_cols = [col for col in merged_df.columns if col.startswith('EER_EP')]
        usa_fuel_features = merged_df[usa_fuel_cols].copy()
        for col in usa_fuel_features.columns:
            usa_fuel_features[col] = pd.to_numeric(usa_fuel_features[col], errors='coerce')

        # 3. Identify 3 distinct Malaysia fuel price series
        malaysia_fuel_cols = [
            'ron95', 'ron97', 'diesel', 'diesel_eastmsia' # Added 'diesel_eastmsia'
        ]
        malaysia_fuel_cols_present = [col for col in malaysia_fuel_cols if col in merged_df.columns]
        malaysia_fuel_features = merged_df[malaysia_fuel_cols_present].copy()
        for col in malaysia_fuel_features.columns:
            malaysia_fuel_features[col] = pd.to_numeric(malaysia_fuel_features[col], errors='coerce')

        # NEW: Extract Brent, WTI, and OPEC as global crude oil features
        global_crude_oil_cols = ['Brent', 'WTI', 'OPEC'] # MODIFIED: Added 'OPEC'
        global_crude_oil_cols_present = [col for col in global_crude_oil_cols if col in merged_df.columns]
        global_crude_oil_features = merged_df[global_crude_oil_cols_present].copy()
        for col in global_crude_oil_features.columns:
            global_crude_oil_features[col] = pd.to_numeric(global_crude_oil_features[col], errors='coerce')

        # 4. Extract 'USD_SDR_USA' and 'USD_SDR_MYS' columns
        usd_sdr_usa = pd.Series(dtype=float)
        if 'USD_SDR_USA' in merged_df.columns:
            usd_sdr_usa = merged_df['USD_SDR_USA'].copy()
        else:
            print("Warning: 'USD_SDR_USA' not found in merged_df. Initializing with NaNs.")
            usd_sdr_usa = pd.Series(index=merged_df.index, dtype=float, name='USD_SDR_USA') # Add name to series

        usd_sdr_mys = pd.Series(dtype=float)
        if 'USD_SDR_MYS' in merged_df.columns:
            usd_sdr_mys = merged_df['USD_SDR_MYS'].copy()
        else:
            print("Warning: 'USD_SDR_MYS' not found in merged_df. Initializing with NaNs.")
            usd_sdr_mys = pd.Series(index=merged_df.index, dtype=float, name='USD_SDR_MYS') # Add name to series

        # 5. Combine USA features
        usa_features = pd.concat([usa_fuel_features, usd_sdr_usa], axis=1)

        # 6. Combine Malaysia features
        malaysia_features = pd.concat([malaysia_fuel_features, usd_sdr_mys], axis=1)

        # 7. Apply forward-filling to handle missing values for all feature sets
        if not usa_features.empty:
            usa_features.ffill(inplace=True)
            usa_features.bfill(inplace=True) # Backfill any remaining NaNs at the beginning
        else:
            print("Warning: usa_features is empty, skipping ffill.")

        if not malaysia_features.empty:
            malaysia_features.ffill(inplace=True)
            malaysia_features.bfill(inplace=True) # Backfill any remaining NaNs at the beginning
        else:
            print("Warning: malaysia_features is empty, skipping ffill.")

        if not global_crude_oil_features.empty:
            global_crude_oil_features.ffill(inplace=True)
            global_crude_oil_features.bfill(inplace=True) # Backfill any remaining NaNs at the beginning
        else:
            print("Warning: global_crude_oil_features is empty, skipping ffill.")

        print("USA Features Head:")
        print(usa_features.head())
        print("\nMalaysia Features Head:")
        print(malaysia_features.head())
        print("\nGlobal Crude Oil Features Head:")
        print(global_crude_oil_features.head())

        # Convert to PyTorch tensors
        usa_features_tensor = torch.tensor(usa_features.values, dtype=torch.float32) if not usa_features.empty else torch.empty(0, usa_features.shape[1] if usa_features.columns.size > 0 else 0, dtype=torch.float32)
        malaysia_features_tensor = torch.tensor(malaysia_features.values, dtype=torch.float32) if not malaysia_features.empty else torch.empty(0, malaysia_features.shape[1] if malaysia_features.columns.size > 0 else 0, dtype=torch.float32)
        global_crude_oil_tensor = torch.tensor(global_crude_oil_features.values, dtype=torch.float32) if not global_crude_oil_features.empty else torch.empty(0, global_crude_oil_features.shape[1] if global_crude_oil_features.columns.size > 0 else 0, dtype=torch.float32)

        print("\nUSA Features Tensor shape:", usa_features_tensor.shape)
        print("Malaysia Features Tensor shape:", malaysia_features_tensor.shape)
        print("Global Crude Oil Features Tensor shape:", global_crude_oil_tensor.shape)

        return usa_features_tensor, malaysia_features_tensor, global_crude_oil_tensor

    def mergreddata(self):
        print("Starting mergreddata process...")
        AAAA = self.Brent()
        BBBB = self.WTI()
        CCCC = self.OPEC()
        DDDD = self.USAFuel_Daily()
        EEEE = self.SDR_XDC()
        FFFF = self.Malaysiafuelprice() # Now calling the enhanced Malaysiafuelprice method


        dataframes_to_concat = []
        if EEEE is not None and not EEEE.empty: # Check if EEEE (SDR data) is not None and not empty
            # Now EEEE is guaranteed to be a DataFrame with a 'value' column
            if 'value' in EEEE.columns: # Changed 'Value' to 'value' as per to_frame() behavior
                try:
                    # Corrected slicing and droplevel to match actual EEEE MultiIndex levels: COUNTRY, INDICATOR, TYPE_OF_TRANSFORMATION, FREQUENCY, TIME_PERIOD
                    if 'USA' in EEEE.index.get_level_values('COUNTRY').unique() and all(level in EEEE.index.names for level in ['COUNTRY', 'INDICATOR', 'TYPE_OF_TRANSFORMATION', 'FREQUENCY', 'TIME_PERIOD']):
                        sdr_usa = EEEE.loc[('USA', 'XDR_XDC', 'PA_RT', 'M', slice(None)), 'value'] \
                                    .droplevel(level=['COUNTRY', 'INDICATOR', 'TYPE_OF_TRANSFORMATION', 'FREQUENCY']) \
                                    .rename('USD_SDR_USA')
                        # Convert monthly index to datetime objects and then deduplicate before resampling
                        sdr_usa.index = pd.to_datetime(sdr_usa.index, format='%Y-M%m').to_period('M').to_timestamp('M')
                        sdr_usa = sdr_usa[~sdr_usa.index.duplicated(keep='first')] # Deduplicate after datetime conversion
                        sdr_usa = sdr_usa.resample('D').ffill()
                        dataframes_to_concat.append(sdr_usa)
                    else:
                        print("Warning: 'USA' or required index levels not found in SDR_XDC data for USD_SDR_USA.")
                except KeyError as ke:
                    print(f"Warning: Issue with SDR_XDC data for USD_SDR_USA: {ke}")
                except Exception as e:
                    print(f"An error occurred processing SDR_USA data: {e}")

                try:
                    # Corrected slicing and droplevel to match actual EEEE MultiIndex levels: COUNTRY, INDICATOR, TYPE_OF_TRANSFORMATION, FREQUENCY, TIME_PERIOD
                    if 'MYS' in EEEE.index.get_level_values('COUNTRY').unique() and all(level in EEEE.index.names for level in ['COUNTRY', 'INDICATOR', 'TYPE_OF_TRANSFORMATION', 'FREQUENCY', 'TIME_PERIOD']):
                        sdr_mys = EEEE.loc[('MYS', 'XDR_XDC', 'PA_RT', 'M', slice(None)), 'value'] \
                                    .droplevel(level=['COUNTRY', 'INDICATOR', 'TYPE_OF_TRANSFORMATION', 'FREQUENCY']) \
                                    .rename('USD_SDR_MYS')
                        # Convert monthly index to datetime objects and then deduplicate before resampling
                        sdr_mys.index = pd.to_datetime(sdr_mys.index, format='%Y-M%m').to_period('M').to_timestamp('M')
                        sdr_mys = sdr_mys[~sdr_mys.index.duplicated(keep='first')] # Deduplicate after datetime conversion
                        sdr_mys = sdr_mys.resample('D').ffill()
                        dataframes_to_concat.append(sdr_mys)
                    else:
                        print("Warning: 'MYS' or required index levels not found in SDR_XDC data for USD_SDR_MYS.")
                except KeyError as ke:
                    print(f"Warning: Issue with SDR_XDC data for USD_SDR_MYS: {ke}")
                except Exception as e:
                    print(f"An error occurred processing SDR_MYS data: {e}")

            else:
                print("Warning: 'value' column not found in SDR_XDC DataFrame.")
        else:
            print("Warning: SDR_XDC data is empty or None. Proceeding without SDR_XDC data due to persistent API issues.")


        # if CCCC is not None and not CCCC.empty and 'OPEC' in CCCC.columns: # REMOVED: OPEC is now handled in global_crude_oil_features
        #     dataframes_to_concat.append(CCCC['OPEC'])
        if AAAA is not None and not AAAA.empty and 'Brent' in AAAA.columns:
            dataframes_to_concat.append(AAAA['Brent'])
        if BBBB is not None and not BBBB.empty and 'WTI' in BBBB.columns:
            dataframes_to_concat.append(BBBB['WTI'])
        if CCCC is not None and not CCCC.empty and 'OPEC' in CCCC.columns: # ADDED: Ensure OPEC data is still available for merging in global_crude_oil_features
            dataframes_to_concat.append(CCCC['OPEC'])
        if DDDD is not None and not DDDD.empty:
            dataframes_to_concat.append(DDDD)
        if FFFF is not None and not FFFF.empty: # Check if Malaysia fuel data is not empty
            dataframes_to_concat.append(FFFF)

        if dataframes_to_concat:
            # Convert all dataframe indices to daily frequency before concatenation to avoid issues
            for i, df_to_convert in enumerate(dataframes_to_concat):
                if isinstance(df_to_convert, pd.Series):
                    if not isinstance(df_to_convert.index, pd.DatetimeIndex):
                        df_to_convert.index = pd.to_datetime(df_to_convert.index)
                elif isinstance(df_to_convert, pd.DataFrame):
                    if not isinstance(df_to_convert.index, pd.DatetimeIndex):
                        df_to_convert.index = pd.to_datetime(df_to_convert.index)


            mergreddata1 = pd.concat(dataframes_to_concat, axis=1, join='outer')

            if not isinstance(mergreddata1.index, pd.DatetimeIndex):
                 mergreddata1.index = pd.to_datetime(mergreddata1.index)

            mergreddata1.sort_index(inplace=True);

            output_csv_path = os.path.join(self.merged_data_path, 'Cleaneddata.csv')
            print(f"Attempting to save merged data to: {output_csv_path}")

            try:
                # Save the DataFrame to CSV *including* the index
                mergreddata1.to_csv(output_csv_path, index=True) # Changed index=False to index=True
                print(f"Successfully saved merged data to {output_csv_path} including index.")

                # Verify if the file was actually created
                if os.path.exists(output_csv_path):
                    print(f"Verification: File exists at {output_csv_path}")
                    print(f"File size: {os.path.getsize(output_csv_path)} bytes")
                else:
                    print(f"Verification: File does NOT exist at {output_csv_path}")


            except Exception as e:
                print(f"Error saving merged Fuel data: {e}")

            recentdatadate = mergreddata1.index[-1]
            print(recentdatadate)
            print(mergreddata1)

            return mergreddata1
        else:
            print("No dataframes available for concatenation.")
            return pd.DataFrame()


# Now, re-run the data fetching and saving step
B = DataFetcher()
merged_df = B.mergreddata()

# Call the new method to prepare country features
usa_features_tensor, malaysia_features_tensor, global_crude_oil_tensor = B.prepare_country_features(merged_df)


# After successfully saving the CSV with the index, re-run the _load_data and get_graph_observation test
try:
    print("\nAttempting to load data and test get_graph_observation after saving CSV with index:")
    # Note: FuelpriceenvfeatureGraph is not defined in the current notebook state.
    # Assuming it's part of a larger context or will be defined later.
    # For now, this part might raise an error if FuelpriceenvfeatureGraph is not available.
    # fuel_env_graph = FuelpriceenvfeatureGraph() # This will call the updated _load_data
    # graph_data, num_nodes, num_edges = fuel_env_graph.get_graph_observation()

    # print("\nTesting get_graph_observation:")
    # print(f"Graph Data Object: {graph_data}")
    # print(f"Number of Nodes: {num_nodes}")
    # print(f"Number of Edges: {num_edges}")
    # if hasattr(graph_data, 'x'):
    #     print(f"Node Features shape (x): {graph_data.x.shape}")
    # if hasattr(graph_data, 'edge_index'):
    #     print(f"Edge Index shape: {graph_data.edge_index.shape}")
    # if hasattr(graph_data, 'graph_attributes'):
    #     print(f"Graph Attributes shape: {graph_data.graph_attributes.shape}")
    # else:
    #     print("Graph attributes not found in the Data object.")

    # # Verify device
    # print(f"Node features device: {usa_features_tensor.device}") # Changed to usa_features_tensor
    # print(f"Edge index device: {malaysia_features_tensor.device}") # Changed to malaysia_features_tensor
    # if usa_features_tensor.numel() > 0 and usa_features_tensor.device:
    #     print(f"Node features device: {usa_features_tensor.device}")
    # if malaysia_features_tensor.numel() > 0 and malaysia_features_tensor.device:
    #     print(f"Node features device: {malaysia_features_tensor.device}")

except Exception as e:
    print(f"Error during testing get_graph_observation: {e}")

Starting mergreddata process...
            Brent
Date             
2003-01-02  30.32
2003-01-03  31.43
2003-01-04    NaN
2003-01-05    NaN
2003-01-06  31.43
...           ...
2026-03-05  88.59
2026-03-06  95.74
2026-03-07    NaN
2026-03-08    NaN
2026-03-09  94.35

[8468 rows x 1 columns]
              WTI
Date             
2003-01-02  31.97
2003-01-03  33.26
2003-01-04    NaN
2003-01-05    NaN
2003-01-06  32.29
...           ...
2026-03-05  80.88
2026-03-06  90.77
2026-03-07    NaN
2026-03-08    NaN
2026-03-09  94.65

[8468 rows x 1 columns]
--- Debug: After initial CSV read ---
Raw data shape: (5983, 3)
Raw data head:
                0              1   2
0  Attribute:data  Attribute:val NaN
1        1/2/2003          30.05 NaN
2        1/3/2003          30.83 NaN
3        1/6/2003          30.71 NaN
4        1/7/2003          29.72 NaN
--- Debug: After column selection and naming ---
All OPEC data shape: (5983, 2)
All OPEC data head:
             Date           OPEC
0  Attribute:dat

value
COUNTRY INDICATOR TYPE_OF_TRANSFORMATION FREQUENCY TIME_PERIOD          
MYS     XDR_XDC   PA_RT                  M         2010-M01     0.189325
                                                   2010-M02     0.190342
                                                   2010-M03     0.196866
                                                   2010-M04     0.205575
                                                   2010-M05     0.207700


Unique countries in fetched data: ['MYS', 'USA']
Unique indicators in fetched data: ['XDR_XDC']
Unique transformation types in fetched data: ['PA_RT']
Unique frequencies in fetched data: ['M']
Fetching Malaysia fuel price data...
Successfully processed Malaysia fuel price data.
Fuel_Type   diesel  diesel_eastmsia  ron95  ron97
Date                                             
2017-03-30    2.11             2.11   2.13   2.41
2017-03-31    2.11             2.11   2.13   2.41
2017-04-01    2.11             2.11   2.13   2.41
2017-04-02    2.11             2.11   2.13   2.41
2017-04-03    2.11             2.11   2.13   2.41
Attempting to save merged data to: /content/drive/MyDrive/deep learning codes/EIAAPI_DOWNLOAD/solutions/mergedata/Cleaneddata.csv
Successfully saved merged data to /content/drive/MyDrive/deep learning codes/EIAAPI_DOWNLOAD/solutions/mergedata/Cleaneddata.csv including index.
Verification: File exists at /content/drive/MyDrive/deep learning codes/EIAAPI_DOWNLOAD/soluti